In [138]:
import numpy as np
import pandas as pd

In [139]:
anime = pd.read_csv('anime.csv')
rating = pd.read_csv('rating.csv')

In [140]:
anime.head(50)

,anime_id,name,genre,type,episodes,rating,members
0,32281,Kimi no Na wa.,"Drama, Romance, School, Supernatural",Movie,1,9.37,200630
1,5114,Fullmetal Alchemist: Brotherhood,"Action, Adventure, Drama, Fantasy, Magic, Mili...",TV,64,9.26,793665
2,28977,Gintama°,"Action, Comedy, Historical, Parody, Samurai, S...",TV,51,9.25,114262
3,9253,Steins;Gate,"Sci-Fi, Thriller",TV,24,9.17,673572
4,9969,Gintama&#039;,"Action, Comedy, Historical, Parody, Samurai, S...",TV,51,9.16,151266
5,32935,Haikyuu!!: Karasuno Koukou VS Shiratorizawa Ga...,"Comedy, Drama, School, Shounen, Sports",TV,10,9.15,93351
6,11061,Hunter x Hunter (2011),"Action, Adventure, Shounen, Super Power",TV,148,9.13,425855
7,820,Ginga Eiyuu Densetsu,"Drama, Military, Sci-Fi, Space",OVA,110,9.11,80679
8,15335,Gintama Movie: Kanketsu-hen - Yorozuya yo Eien...,"Action, Comedy, Historical, Parody, Samurai, S...",Movie,1,9.10,72534
9,15417,Gintama&#039;: Enchousen,"Action, Comedy, Historical, Parody, Samurai, S...",TV,13,9.11,81109


In [141]:
anime['genre']

0                     Drama, Romance, School, Supernatural
1        Action, Adventure, Drama, Fantasy, Magic, Mili...
2        Action, Comedy, Historical, Parody, Samurai, S...
3                                         Sci-Fi, Thriller
4        Action, Comedy, Historical, Parody, Samurai, S...
                               ...                        
12289                                               Hentai
12290                                               Hentai
12291                                               Hentai
12292                                               Hentai
12293                                               Hentai
Name: genre, Length: 12294, dtype: object

In [142]:
rating.head(1)

,user_id,anime_id,rating
0,1,20,-1


In [143]:
anime.shape

(12294, 7)

In [144]:
rating.shape

(7813737, 3)

In [145]:
anime.info()
rating.info()

anime.isnull().sum()
rating.isnull().sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12294 entries, 0 to 12293
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   anime_id  12294 non-null  int64  
 1   name      12294 non-null  object 
 2   genre     12232 non-null  object 
 3   type      12269 non-null  object 
 4   episodes  12294 non-null  object 
 5   rating    12064 non-null  float64
 6   members   12294 non-null  int64  
dtypes: float64(1), int64(2), object(4)
memory usage: 672.5+ KB
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7813737 entries, 0 to 7813736
Data columns (total 3 columns):
 #   Column    Dtype
---  ------    -----
 0   user_id   int64
 1   anime_id  int64
 2   rating    int64
dtypes: int64(3)
memory usage: 178.8 MB


user_id     0
anime_id    0
rating      0
dtype: int64

In [146]:
rating = rating[rating['rating'] != -1]

In [147]:
df = rating.merge(anime, on='anime_id')

In [148]:
df.head(1)

,user_id,anime_id,rating_x,name,genre,type,episodes,rating_y,members
0,1,8074,10,Highschool of the Dead,"Action, Ecchi, Horror, Supernatural",TV,12,7.46,535892


In [149]:
df.shape

(6337239, 9)

In [150]:
df = df.rename(columns={
    'rating_x':'user_rating',
    'rating_y':'anime_rating'
})

In [151]:
df.drop(['type', 'episodes'], axis=1, inplace=True)

In [152]:
df= df.drop(columns=['members'])

In [153]:
df.head(1)

,user_id,anime_id,user_rating,name,genre,anime_rating
0,1,8074,10,Highschool of the Dead,"Action, Ecchi, Horror, Supernatural",7.46


In [154]:
df.isnull().sum()

user_id          0
anime_id         0
user_rating      0
name             0
genre           88
anime_rating     5
dtype: int64

In [155]:
df.dropna(inplace=True)

In [156]:
df.isnull().sum()

user_id         0
anime_id        0
user_rating     0
name            0
genre           0
anime_rating    0
dtype: int64

In [157]:
df.duplicated().sum()

np.int64(1)

In [120]:
df.drop_duplicates(inplace=True)

In [121]:
df.duplicated().sum()

np.int64(0)

In [122]:
df['user_id'].nunique()

69600

In [123]:
df['anime_id'].nunique()

9892

In [127]:
df['genre'].isnull().sum()

np.int64(0)

In [124]:
df['genre']=df['genre'].apply(lambda x:x.split(', '))

In [125]:
def collapse(L):
    return [i.replace(" ","") for i in L]
df['genre'] = df['genre'].apply(collapse)

In [126]:
df['tags'] = df['genre'].apply(lambda x: " ".join(x))

In [128]:
df['tags'] = df['tags'].str.replace('Sci-Fi', 'SciFi')

In [129]:
df['tags'].head()

0                   Action Ecchi Horror Supernatural
1           Comedy Demons Ecchi Harem Romance School
2              Action Adventure Fantasy Game Romance
3    Action Comedy Demons Ecchi Harem Romance School
4                       Comedy School Shounen Sports
Name: tags, dtype: object

In [130]:
df['tags'] = df['tags'].apply(lambda x: x.lower())

In [181]:
##CONTENT BASED RECOMMENDATION ON GENRE

In [159]:
anime_df = df[['anime_id','name','genre']].drop_duplicates()

In [160]:
anime_df.shape

(9892, 3)

In [161]:
anime_df.dropna(inplace=True)

anime_df['genre'] = anime_df['genre'].apply(lambda x: x.split(', '))

def collapse(L):
    return [i.replace(" ","") for i in L]

anime_df['genre'] = anime_df['genre'].apply(collapse)

anime_df['tags'] = anime_df['genre'].apply(lambda x: " ".join(x))

anime_df['tags'] = anime_df['tags'].str.replace('-', '', regex=False)

anime_df['tags'] = anime_df['tags'].str.lower()

In [162]:
from sklearn.feature_extraction.text import CountVectorizer
cv = CountVectorizer(max_features=5000, stop_words='english')
vectors = cv.fit_transform(anime_df['tags']).toarray()

In [163]:
cv.get_feature_names_out()

array(['action', 'adventure', 'cars', 'comedy', 'dementia', 'demons',
       'drama', 'ecchi', 'fantasy', 'game', 'harem', 'hentai',
       'historical', 'horror', 'josei', 'kids', 'magic', 'martialarts',
       'mecha', 'military', 'music', 'mystery', 'parody', 'police',
       'psychological', 'romance', 'samurai', 'school', 'scifi', 'seinen',
       'shoujo', 'shoujoai', 'shounen', 'shounenai', 'sliceoflife',
       'space', 'sports', 'supernatural', 'superpower', 'thriller',
       'vampire', 'yaoi', 'yuri'], dtype=object)

In [164]:
from sklearn.metrics.pairwise import cosine_similarity

In [165]:
similarity = cosine_similarity(vectors)

In [166]:
similarity[0]

array([1.        , 0.20412415, 0.2236068 , ..., 0.        , 0.        ,
       0.        ])

In [200]:
def recommend_genre(anime):
    anime = anime.strip()
    anime_index = anime_df[anime_df['name']==anime].index[0]
    distances=similarity[anime_index]
    anime_list = sorted(list(enumerate(distances)),key=lambda x:x[1],reverse=True)[1:6]
    recommendations=[]
    for i in anime_list:
        recommendations.append(
            anime_df.iloc[i[0]]['name']
        )

    return recommendations
    ##print(anime_df.iloc[i[0]]['name'])

In [201]:
recommend_genre("Death Note")

['Death Note Rewrite',
 'Mousou Dairinin',
 'Higurashi no Naku Koro ni Kai',
 'Higurashi no Naku Koro ni',
 'Higurashi no Naku Koro ni Rei']

In [180]:
##COLLABERATIVE FILTERING

In [174]:
df

,user_id,anime_id,user_rating,name,genre,anime_rating
0,1,8074,10,Highschool of the Dead,"Action, Ecchi, Horror, Supernatural",7.46
1,1,11617,10,High School DxD,"Comedy, Demons, Ecchi, Harem, Romance, School",7.70
2,1,11757,10,Sword Art Online,"Action, Adventure, Fantasy, Game, Romance",7.83
3,1,15451,10,High School DxD New,"Action, Comedy, Demons, Ecchi, Harem, Romance,...",7.87
4,2,11771,10,Kuroko no Basket,"Comedy, School, Shounen, Sports",8.46
...,...,...,...,...,...,...
6337234,73515,16512,7,Devil Survivor 2 The Animation,"Action, Demons, Supernatural",7.06
6337235,73515,17187,9,Ghost in the Shell: Arise - Border:1 Ghost Pain,"Mecha, Police, Psychological, Sci-Fi",7.64
6337236,73515,22145,10,Kuroshitsuji: Book of Circus,"Comedy, Demons, Fantasy, Historical, Shounen, ...",8.37
6337237,73516,790,9,Ergo Proxy,"Mystery, Psychological, Sci-Fi",8.03


In [175]:
rating_counts = df.groupby('name')['user_rating'].count()

In [177]:
popular_anime=rating_counts[rating_counts>=50].index
filter_df=df[df['name'].isin(popular_anime)]

In [178]:
pt = filter_df.pivot_table(index='name',columns='user_id',values='user_rating')

In [179]:
pt.fillna(0,inplace=True)

In [182]:
from scipy.sparse import csr_matrix
anime_sparse = csr_matrix(pt)

In [183]:
from sklearn.neighbors import NearestNeighbors

In [184]:
model = NearestNeighbors(metric='cosine', algorithm= 'brute')

In [185]:
model.fit(anime_sparse)

NearestNeighbors(algorithm='brute', metric='cosine')

In [206]:
import numpy as np
def recommend_collab(anime_name):
    anime_index = np.where(pt.index==anime_name)[0][0]
    distances,suggestions = model.kneighbors(pt.iloc[anime_index,:].values.reshape(1,-1),n_neighbors=6)
    recommendation=[]
    for i in suggestions[0][1:]:
        recommendation.append(
            pt.index[i]
        )
    return recommendation

In [207]:
recommend_collab("Death Note")

['Code Geass: Hangyaku no Lelouch',
 'Code Geass: Hangyaku no Lelouch R2',
 'Elfen Lied',
 'Shingeki no Kyojin',
 'Fullmetal Alchemist: Brotherhood']

In [208]:
def recommend_hybrid(anime):
    content_results = recommend_genre(anime)
    collab_results = recommend_collab(anime)
    hybrid = content_results + collab_results
    hybrid = list(dict.fromkeys(hybrid))
    return hybrid[:10]

In [209]:
recommend_hybrid("Death Note")

['Death Note Rewrite',
 'Mousou Dairinin',
 'Higurashi no Naku Koro ni Kai',
 'Higurashi no Naku Koro ni',
 'Higurashi no Naku Koro ni Rei',
 'Code Geass: Hangyaku no Lelouch',
 'Code Geass: Hangyaku no Lelouch R2',
 'Elfen Lied',
 'Shingeki no Kyojin',
 'Fullmetal Alchemist: Brotherhood']